In [1]:
import sys
dir_git = "/Users/usuario/git/sisepuede"
if dir_git not in sys.path:
    sys.path.append(dir_git)


import importlib
import matplotlib.pyplot as plt
import numpy as np
import os, os.path
import pandas as pd
import pathlib
import sisepuede.core.support_classes as sc
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.utilities.data_support._elasticities as elast
import sisepuede.utilities._toolbox as sf
import utils.common_data_needs as cdn
import warnings
warnings.filterwarnings("ignore")

from typing import *

plt.style.use("dark_background", )



/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:533: UserWarning: Path '/Users/usuario/git/sisepuede/sisepuede/out/sisepuede_run_2025-10-26T11;41;25.002512' not found. It will not be created.
  warnings.warn(msg)
/Users/usuario/git/sisepuede/sisepuede/core/model_attributes.py:6830: UserWarning: 

                        MISSIONSEARCHNOTE: As of 2023-10-06, there is a temporary solution 
                        implemeted in ModelAttributes.get_variable_to_simplex_group_dictionary() 
                        to ensure that transition probability rows are enforced on a simplex.
                        
                        
                        FIX THIS ASAP TO DERIVE PROPERLY.
                        
                        
  warnings.warn(
/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:533: UserWarning: Path '/Users/usuario/git/sisepuede/sisepuede/out/sisepuede_run_2025-10-26T11;41;25.357747' not found. It will not be created.
  warnings.warn(msg)
/Use

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Precompiling NemoMod...
Info Given NemoMod was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   1619.3 ms  ? NemoMod
[ Info: Precompiling NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72] 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing NemoMod [a3c327a0-d2f0-11e8-37fd-d12fd35c3c72].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
/Users/usuario/git/sisepuede/sisepuede/utilities/_toolbox.py:2633: UserWarning: Warning passed from optional_log: Successfully initialized JuMP optimizer from solver module HiGHS..
  warnings.warn(f"Warning passed from optional_log: {msg}.")


# Initialize some SISEPUEDE components

In [2]:
dict_ssp = cdn._setup_sisepuede_elements()

matt = dict_ssp.get("model_attributes", )
models = dict_ssp.get("models", )
regions = dict_ssp.get("regions", )
time_periods = dict_ssp.get("time_periods", )

# setup region
_REGION_NAME = "uganda"
_REGION_ISO = regions.return_region_or_iso(_REGION_NAME, return_type = "iso")

# set year of survey data
_YEAR_START = 2015
_YEAR_SURVEY = 2021
_YEAR_TARGET = 2100
_DF_YEARS = cdn.spawn_years_space_df((_YEAR_START, _YEAR_TARGET + 1))


##################################################
#    initialize a global dict for populations    #
##################################################

_ATTRIBUTE_TABLE_LVST = matt.get_attribute_table(matt.subsec_name_lvst, )


## Use data from [UN SDG](https://www.sdg6data.org/en/country-or-area/Uganda) to estimate some treatment pathways

- Their data system is stupid, and it is very difficult to obtain raw data


###  Change in sanitation over time

- From 2000-2024:
     - Safely managed increases from 13% to 20% from 2000-2024
     - Improved increases from 29% to 38%
![Change in improved over time](./input_data/unsdg/unsdg_6.2.1a_timeseries_29-38_and_13-20.png)


###  Sanitation in 2024

![Change in improved over time](./input_data/unsdg/unsdg_6.2.1a.png)


###  Treatment characteristics

- Note that [Bugolobi Wastewater Treatment Plant](https://en.wikipedia.org/wiki/Bugolobi_Wastewater_Treatment_Plant#:~:text=Bugoloobi%20Wastewater%20Treatment%20Plant%20(BWTP,m3)%20of%20wastewater%20daily.) has gas capture for waste to energy and sludge recollection for fertilizer (est to cover 25% of Kampala's daily needs)
- We set 25% of treatment to recollection for these purposes in the baseline


In [27]:
# attribute tables
_ATTR_TRWW = matt.get_attribute_table(matt.subsec_name_trww, )
_ATTR_WALI = matt.get_attribute_table(matt.subsec_name_wali, )

# model variables TRWW
_MODVAR_TRWW_RF_BIOGAS = matt.get_variable(models.model_circecon.modvar_trww_rf_biogas_recovered, )

# model variables WALI
_MODVAR_WALI_AAER = matt.get_variable(models.model_circecon.modvar_wali_treatpath_advanced_aerobic, )
_MODVAR_WALI_AANER = matt.get_variable(models.model_circecon.modvar_wali_treatpath_advanced_anaerobic, )
_MODVAR_WALI_LIMP = matt.get_variable(models.model_circecon.modvar_wali_treatpath_latrine_improved, )
_MODVAR_WALI_LUNIMP = matt.get_variable(models.model_circecon.modvar_wali_treatpath_latrine_unimproved, )
_MODVAR_WALI_SAER = matt.get_variable(models.model_circecon.modvar_wali_treatpath_secondary_aerobic, )
_MODVAR_WALI_SANER = matt.get_variable(models.model_circecon.modvar_wali_treatpath_secondary_anaerobic, )
_MODVAR_WALI_SEP = matt.get_variable(models.model_circecon.modvar_wali_treatpath_septic, )
_MODVAR_WALI_UN_NOSEW = matt.get_variable(models.model_circecon.modvar_wali_treatpath_untreated_no_sewerage, )
_MODVAR_WALI_UN_SEW = matt.get_variable(models.model_circecon.modvar_wali_treatpath_untreated_with_sewerage, )
_MODVARS_WALI_FRACS = [
    _MODVAR_WALI_AAER,
    
]

# get limited indicators from SDG data
df_indicators = pd.read_csv(
    cdn._PATH_INPUTS.joinpath("unsdg", "unsdg_6.2.1a_table.csv")
)

#
df_indicators[
    df_indicators["region"].isin(["Uganda"])
].transpose()



,226
region,Uganda
year,2022-2024
Safely managed service,20.36
At least basic service,NaN
Basic service,24.0
Limited service,17.14
Unimproved,58.03
Open defecation,3.53


In [107]:
# assume that safely managed is secondary or tertiary treatment in urban
# septic in rural
# basic is sewerage in urban, improved latrine in rural
# unimproved we use sewerage in urban, unimproved latrine in rural
# open is no treatment

# _MODVAR_WALI_UN_NOSEW

# assigned from looking at figure for 2024-2020
dict_urban = {
    _MODVAR_WALI_AANER: 0.06, 
    _MODVAR_WALI_SANER: 0.18,
    _MODVAR_WALI_LIMP: 0.32, # basic?
    _MODVAR_WALI_LUNIMP: 0.405,
    #_MODVAR_WALI_UN_SEW: 0.32,
}

# rest is open no sewerage
dict_rural = {
    _MODVAR_WALI_SEP: 0.18, 
    _MODVAR_WALI_LIMP: 0.20,
    _MODVAR_WALI_LUNIMP: 0.56,
}

# industrial, we don't know. Assume similar to urban, but without latrines
dict_ind = {
    _MODVAR_WALI_AANER: 0.06, 
    _MODVAR_WALI_SANER: 0.18,
    _MODVAR_WALI_UN_SEW: 0.725,
}


dict_fracs_2024 = {
    "ww_industrial": dict_ind,
    "ww_domestic_rural": dict_rural,
    "ww_domestic_urban": dict_urban,
}

# build new data frame
df_new = {
    time_periods.field_year: [2024],
}
for cat, dict_pathways in dict_fracs_2024.items():
    for modvar, frac in  dict_pathways.items():
        field = modvar.build_fields(
            category_restrictions = cat,
        )

        # update
        df_new.update({field: [frac]})

df_new = pd.DataFrame(df_new)


##  NOW, BUILD SCALAR BACK FROM 2024-2000

# use the scalar from the increasing figure for safe access
y0 = 2000
y1 = 2024
val_0 = 0.13
val_1 = 0.2

field_ramp = "ramp"
vec_ramp = np.linspace(val_0, val_1, y1 - y0 + 1)/val_1

df_ramp = pd.DataFrame(
    {
        time_periods.field_year: range(y0, y1 + 1),
        field_ramp: vec_ramp,
    }
)


##  NOW, BUILD PATHWAYS

fields_adj = [x for x in df_new.columns if x not in df_ramp.columns]
df_pathways = (
    pd.merge(
        df_ramp,
        df_new,
        how = "left",
    )
    .bfill()
    .ffill()
)


# adjust
df_pathways[fields_adj] = sf.do_array_mult(
    df_pathways[fields_adj].to_numpy(),
    df_pathways[field_ramp].to_numpy()
)

modvars_trww = [
    v.get("treatment_fraction") for v in 
    models
    .model_circecon
    .dict_trww_categories_to_wali_fraction_variables
    .values()
]

all_fields = []
# then, for each category, get fields
for cat in dict_fracs_2024.keys():
    fields = []
    for modvar in modvars_trww:
        modvar = matt.get_variable(modvar, )
        fields.append(
            modvar.build_fields(category_restrictions = cat, )
        )
        
    all_fields.extend(fields, )
        
    # get fraction to assign to unsewered/untreated (open def.)
    fields_ext = [x for x in df_pathways.columns if x in fields]
    vec_new = 1 - df_pathways[fields_ext].sum(axis = 1)
    if vec_new.min() < 0:
        raise RuntimeError(f"In unsewered/treated water... Sum of specified pathways exceeds 1")

    
    # assign to unsewered
    field_unsewered = _MODVAR_WALI_UN_NOSEW.build_fields(
        category_restrictions = cat,
    )

    if field_unsewered in fields_ext:
        raise RuntimeError(f"Unsewered fraction already specified: check dictionaries")

    
    df_pathways[field_unsewered] = vec_new


# finally, extract all fields needed
all_fields_remaining = [x for x in all_fields if x not in df_pathways]
df_pathways[all_fields_remaining] = 0
df_pathways = df_pathways.drop(
    columns = field_ramp,
)

# 
df_pathways = (
    pd.merge(
        cdn.spawn_years_space_df((y0, 2101), ),
        df_pathways,
        how = "left",
    )
    .bfill()
    .ffill()
)

In [112]:
# export
fn = "wastewater_treatment_pathways.csv"
sf._write_csv(
    df_pathways,
    cdn._PATH_OUTPUTS.joinpath(fn, ),
)


True